In [ ]:
import pandas as pd
import os
import json
import librosa
import torch
import numpy as np
import ast
import pickle

import warnings
warnings.filterwarnings(action='ignore')

# data preprocess

In [ ]:
# 발화자 id별로 dataframe 설정
df_wav=pd.DataFrame()

# 지정된 폴더 경로
directory_path = '/data2/brian/한국인의_주제적응형_말하기평가/training/source'

id_list=[]
wav_file_list=[]
for i in os.listdir(directory_path):
    path = os.path.join(directory_path, i)
    if os.path.isdir(path):  # 디렉터리인지 확인
        for file_no in os.listdir(path):
            if os.path.isdir(os.path.join(path, file_no)):  # 이것도 디렉터리인지 확인
                id_list.append(file_no[:-4])
                wav_file_list.append([os.path.join(path, file_no, x) for x in os.listdir(os.path.join(path, file_no)) if x.endswith('.wav')])  # wav 파일만 추가
                
df_wav['id']=id_list
df_wav['wav_file']=wav_file_list
df_wav=df_wav.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# train/valid/test 분할(0.7/0.15/0.15)
split = 0.7
train_length = int(len(df_wav) * split)
valid_length = int(len(df_wav) * (1 - split) / 2)

train = df_wav[:train_length]
valid = df_wav[train_length:train_length + valid_length]
test = df_wav[train_length + valid_length:]

In [ ]:
# 발화자 source path 리스트로 펼치기
def list_append(x):
  globals()[f'{x}_list']=[]
  for i in x['wav_file']:
    for j in i:
      globals()[f'{x}_list'].append(j)
    
  return globals()[f'{x}_list']

In [ ]:
train_df=pd.DataFrame()
train_df['wav_path']=list_append(train)
train_df['json_path']=train_df['wav_path'].str.replace('source','label').str.replace('wav','json')

valid_df=pd.DataFrame()
valid_df['wav_path']=list_append(valid)
valid_df['json_path']=valid_df['wav_path'].str.replace('source','label').str.replace('wav','json')

test_df=pd.DataFrame()
test_df['wav_path']=list_append(test)
test_df['json_path']=test_df['wav_path'].str.replace('source','label').str.replace('wav','json')


In [ ]:
# json 'words' 정보 불러오기
def words(x):
  words_len_list=[]
  words_list=[]
  start_list=[]
  end_list=[]
  for i in x['json_path']:
    with open(i, 'r') as file:
        data = json.load(file)
        words = data['utterance']['words']
        words_len_list.append(len(words))
        words_list.append([item['name'] for item in words])
        start_list.append([float(item['start']) for item in words])
        end_list.append([float(item['end']) for item in words])
  return words_len_list, words_list, start_list, end_list

In [ ]:
train_df['len'],train_df['words'],train_df['start'],train_df['end']=words(train_df)
valid_df['len'],valid_df['words'],valid_df['start'],valid_df['end']=words(valid_df)
test_df['len'],test_df['words'],test_df['start'],test_df['end']=words(test_df)

In [ ]:
# wav 임베딩
def audio_transformed(x):
  audio_transformed_list=[]
  for i in range(len(x)):
    words_transformed_list=[]
    for j in range(x.loc[i,'len']):
      input_file = x.loc[i,'wav_path']
      start_time = x.loc[i,'start'][j]  # 시작 시간 (초)
      end_time = x.loc[i,'end'][j]  # 종료 시간 (초)

      # 오디오 파일 로드 (모델에 필요한 샘플링 레이트로 조정)
      audio, sr = librosa.load(input_file, sr=16000)

      # 시작 시간과 종료 시간을 샘플 인덱스로 변환
      start_sample = int(start_time * sr)
      end_sample = int(end_time * sr)

      # 지정된 구간의 오디오 샘플 추출
      audio_segment = audio[start_sample:end_sample]

      # 입력 신호에 패딩 추가
      audio_segment = librosa.util.fix_length(audio_segment, size=2048)

      # MFCC 특성 추출 (지정된 구간에 대해서만, n_fft 조정)
      mfcc = librosa.feature.mfcc(y=audio_segment, sr=sr, n_mfcc=13, n_fft=1024)

      # MFCC의 평균 계산 (모든 프레임에 대해)
      mfcc_mean = np.mean(mfcc, axis=1)

      # 평균 MFCC를 (13, 1) 형태로 재구성
      mfcc_mean_reshaped = mfcc_mean.reshape(-1)

      # PyTorch 텐서로 변환
      mfcc_tensor = torch.tensor(mfcc_mean_reshaped, dtype=torch.float32)

      # 모델 입력 준비 완료
      words_transformed_list.append(mfcc_tensor.tolist())

    audio_transformed_list.append(words_transformed_list)
  
  return audio_transformed_list

In [ ]:
train_df['audio']=audio_transformed(train_df)
valid_df['audio']=audio_transformed(valid_df)
test_df['audio']=audio_transformed(test_df)
%time

In [ ]:
#words 임베딩
# GloVe 파일 경로
glove_file = './glove.6B/glove.6B.50d.txt'

# GloVe 단어 임베딩 로드
embeddings_index = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs


# 단어 임베딩 함수 수정
def embed_words(words):
    embeddings = []
    len_embed = 0
    for word in words:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embeddings.append(embedding_vector.tolist())
            len_embed += 1
        else:
            embeddings.append(np.zeros(50).tolist())  # 임베딩이 없는 단어에 대해 0의 리스트 추가 임베딩 차원에 맞게 수정
    return embeddings, len_embed


In [ ]:
# 각 행에 함수 적용 및 결과 할당
result = train_df['words'].apply(lambda x: embed_words(x))
train_df['embedded'] = result.apply(lambda x: x[0])
train_df['len_embedded'] = result.apply(lambda x: x[1])

# 각 행에 함수 적용 및 결과 할당
result = valid_df['words'].apply(lambda x: embed_words(x))
valid_df['embedded'] = result.apply(lambda x: x[0])
valid_df['len_embedded'] = result.apply(lambda x: x[1])

# 각 행에 함수 적용 및 결과 할당
result = test_df['words'].apply(lambda x: embed_words(x))
test_df['embedded'] = result.apply(lambda x: x[0])
test_df['len_embedded'] = result.apply(lambda x: x[1])

%time

In [ ]:
# json 'features' 정보 불러오기
def features(x):
  acoustic_feature=[]
  lexical_feature=[]
  syntactic_feature=[]
  for i in x['json_path']:
    with open(i, 'r') as file:
        data = json.load(file)
        acoustic_feature.append([float(x) for x in list(data['acoustic_feature'].values())])
        lexical_feature.append([float(x) for x in list(data['lexical_feature'].values())])
        syntactic_feature.append([float(x) for x in list(data['syntactic_feature'].values())])
  return acoustic_feature, lexical_feature, syntactic_feature


train_df['acoustic_feature'],train_df['lexical_feature'],train_df['syntactic_feature']=features(train_df)
valid_df['acoustic_feature'],valid_df['lexical_feature'],valid_df['syntactic_feature']=features(valid_df)
test_df['acoustic_feature'],test_df['lexical_feature'],test_df['syntactic_feature']=features(test_df)

In [ ]:
train_df['div']='train'
valid_df['div']='valid'
test_df['div']='test'

In [ ]:
# 문자열을 해당하는 숫자로 매핑하는 함수
def convert_to_number(category):
    mapping = {
        "IG": 1,
        "TL": 2,
        "TM": 3,
        "TH": 4,
        "NA": 5
    }
    return mapping.get(category, 0)  # 매핑에 없는 카테고리는 "Unknown" 처리

# json 'features' 정보 불러오기
def label(x):
  label=[]
  for i in x['json_path']:
    with open(i, 'r') as file:
        data = json.load(file)
        label.append([float(data['rating']['Task_Completion']['rater_final']),
                      float(data['rating']['Delivery']['rater_final']),
                      float(data['rating']['Accuracy']['rater_final']),
                      float(data['rating']['Appropriateness']['rater_final']),
                      int(convert_to_number(data['speaker']['level']['final']))])
  return label

train_df['label']=label(train_df)
valid_df['label']=label(valid_df)
test_df['label']=label(test_df)

In [ ]:
df=pd.DataFrame()

In [ ]:
df=pd.concat([train_df,valid_df])

In [ ]:
df=pd.concat([df,test_df])

In [ ]:
df=df[df['len']!=0]

In [ ]:
df=df.reset_index(drop=True)

In [ ]:
target_length=max(df['len'])

zero_pdding_audio=[]
for i in df['audio']:
    zero_padding_list=[]
    for j in i:
        zero_padding_list.append(j)
    for _ in range((target_length - len(i))):
        zero_padding_list.append([0] * 13)
    zero_pdding_audio.append(zero_padding_list)
df['audio']=zero_pdding_audio

In [ ]:
zero_pdding_audio=[]
for i in df['embedded']:
    zero_padding_list=[]
    for j in i:
        zero_padding_list.append(j)
    for _ in range((target_length - len(i))):
        zero_padding_list.append([0] * 50) #임베딩 차원에 맞게 수정
    zero_pdding_audio.append(zero_padding_list)
df['embedded']=zero_pdding_audio

In [ ]:
torch.tensor(df['audio']).shape

In [ ]:
torch.tensor(df['embedded']).shape

In [ ]:
torch.tensor(df['label']).shape

In [ ]:
df.columns

In [ ]:
df.columns=['wav_path', 'json_path', 'len', 'words', 'start', 'end',
       'audio', 'text', 'len_embedded', 'acoustic_feature',
       'lexical_feature', 'syntactic_feature', 'div', 'labels']

In [ ]:
# standared scaling
def restore_original_structure(scaled_data, original_df):
    start = 0
    restored_data = []
    for lst in original_df[i]:
        end = start + len(lst)
        restored_data.append(list(scaled_data[start:end]))
        start = end
    return restored_data

for i in ['audio','text','acoustic_feature', 'lexical_feature','syntactic_feature']:
    # 전체 데이터를 하나의 배열로 변환
    all_data = np.concatenate(df[i].values)

    # 피처 스케일링 적용
    scaled_all_data = (all_data - all_data.mean()) / all_data.std()

    # 원래 구조로 복원
    df[i] = restore_original_structure(scaled_all_data, df)

In [ ]:
with open('df_scaling_50d_769hr.pkl', 'wb') as file:
    pickle.dump(df, file)

In [ ]:
df_input = pd.DataFrame(columns=['train', 'valid', 'test'], index=['audio', 'text', 'acoustic_feature','lexical_feature','syntactic_feature','labels'])

In [ ]:
"""
#### lexical_feature가 없는 데이터 삭제하는 코드 ####

# Calculate length for each item in 'lexical_feature'
df['length'] = df['lexical_feature'].apply(len)

# Filter out rows where 'length' is 0
df = df[df['length'] > 0]

# Drop the 'length' column if it's no longer needed
df = df.drop(columns=['length'])

df =df.reset_index(drop=True)

#### syntactic_feature가 없는 데이터 삭제하는 코드 ####
df['length'] = df['syntactic_feature'].apply(len)

# Filter out rows where 'length' is 0
df = df[df['length'] > 0]

# Drop the 'length' column if it's no longer needed
df = df.drop(columns=['length'])

df =df.reset_index(drop=True)
"""

In [ ]:
for i in df_input.index:
    for j in df_input.columns:
        print("i: ", i)
        print("j: ", j)
        df_input.loc[i,j]=torch.stack([torch.tensor(row, dtype=torch.float32) for row in df[df['div']==j][i]])


In [ ]:
with open('df_input_scaling_50d_769hr.pkl', 'wb') as file:
    pickle.dump(df_input, file)

In [ ]:
print("최종 사용된 데이터 개수:", len(df))
total_duration = sum([librosa.get_duration(filename=file) for file in df['wav_path']])
total_duration = total_duration / 3600
print("음성 파일의 총 시간:", total_duration, "시간")